In [1]:
import polars as pl
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

import pyreadstat as prs
import pygwalker as pyg

import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt

import json

from tqdm.notebook import tqdm
from itables import init_notebook_mode
init_notebook_mode()

In [2]:
# panel = pd.read_parquet('../data/RLMS_IND_2016_2024_eng.parquet')
# panel

In [3]:
panel, meta = prs.read_dta('/Volumes/MODELS/RLMS_IND_1994_2024_eng.dta')
panel = panel[panel['int_y'] >= 2015]
panel

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [4]:
for col in panel.columns:
    panel[col] = np.where(panel[col] > 88888888, np.nan, panel[col])

In [5]:
variable_value_labels = meta.variable_value_labels
variable_value_labels

{'id_w': {5: ' 1994',
  6: ' 1995',
  7: ' 1996',
  8: ' 1998',
  9: ' 2000',
  10: ' 2001',
  11: ' 2002',
  12: ' 2003',
  13: ' 2004',
  14: ' 2005',
  15: ' 2006',
  16: ' 2007',
  17: ' 2008',
  18: ' 2009',
  19: ' 2010',
  20: ' 2011',
  21: ' 2012',
  22: ' 2013',
  23: ' 2014',
  24: ' 2015',
  25: ' 2016',
  26: ' 2017',
  27: ' 2018',
  28: ' 2019',
  29: ' 2020',
  30: ' 2021',
  31: ' 2022',
  32: ' 2023',
  33: ' 2024'},
 'origsm': {0: 'No, it is not belong to the representative sample: the family moved had been sur',
  1: 'Yes, it is an adress from the representative sample'},
 'region': {1: 'Leningrad Oblast: Volosovkij Rajon',
  9: 'Krasnodar CR',
  10: 'Udmurt ASSR: Glasov CR',
  12: 'Perm Territory: Solikamsk City & Rajon',
  14: 'Kaluzhskaya Oblast: Kuibyshev Rajon',
  33: 'Tambov Oblast: Uvarovo CR',
  39: 'Volgograd Oblast: Rudnjanskij Rajon',
  45: 'Tatarskaja ASSR: Kazan',
  46: 'Kurgan',
  47: 'Orenburg Oblast: Orsk',
  48: 'Chuvashskaya ASSR: Shumerlja CR',
  

In [6]:
ls = []
for key_aux, val_aux in variable_value_labels.items():
    for key, val in val_aux.items():
        ls.append(
            pd.DataFrame({
                'question': [key_aux] * len(val),
                'answer': [key] * len(val),
                'mapping': val
            })
        )
variable_value_labels_df = pd.concat(ls)
variable_value_labels_df = variable_value_labels_df[variable_value_labels_df['answer'] < 88888888.0]
variable_value_labels_df

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [8]:
column_names_to_labels = meta.column_names_to_labels

In [9]:
variable_value_labels_df['label'] = variable_value_labels_df['question'].map(column_names_to_labels)
variable_value_labels_df['mapping'] = variable_value_labels_df['mapping'].str.strip()
variable_value_labels_df

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [10]:
variable_value_labels_df.sort_values('answer', ascending=False).drop_duplicates('answer')

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [11]:
variable_value_labels_df['mapping'].value_counts().to_frame()

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [12]:
variable_value_labels_df = variable_value_labels_df.drop_duplicates()
variable_value_labels_df

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [13]:
cols_replacement = list(variable_value_labels_df['question'].unique())
cols_replacement

['id_w',
 'origsm',
 'region',
 'psu',
 'status',
 'adult',
 'child',
 'marst',
 'occup08',
 'educ',
 'diplom',
 'diplom_1',
 'h4_1',
 'h5',
 'h7_2',
 'h7_2_l',
 'born_m',
 'i1',
 'i2',
 'i3',
 'i9_1',
 'i3_1',
 'i4',
 'i5',
 'i6',
 'i10_1',
 'i11_1',
 'i12_1',
 'i12_2',
 'i12_3',
 'i12_4',
 'i12_5',
 'i12_6',
 'i12_7',
 'i13',
 'i14_0',
 'i14',
 'i16_1',
 'i16_2',
 'i16_3',
 'i17_1',
 'i17_2',
 'i17_3',
 'j1',
 'j1_1_1',
 'j1_1_2',
 'j1_1_3',
 'j1_1_4',
 'j1_1_5',
 'j1_1_6',
 'j1_1_7',
 'j1_1_8',
 'j1_1_9',
 'j1_1_10',
 'j2cod08',
 'j4a_1',
 'j4a_2',
 'j4a_3',
 'j4_2',
 'j4_3',
 'j4_1',
 'j5b',
 'j5_2',
 'j6',
 'j482',
 'j484',
 'j7',
 'j8_1',
 'j8_3',
 'j9',
 'j10_3',
 'j11',
 'j11_1',
 'j11_2',
 'j14',
 'j17',
 'j18_2',
 'j19',
 'j20b',
 'j21a',
 'j21_1_1',
 'j21_1_2',
 'j21_1_3',
 'j21_1_4',
 'j21_1_5',
 'j21_1_6',
 'j21_1_7',
 'j21_1_8',
 'j211_13',
 'j21_1_9',
 'j211_10',
 'j211_12',
 'j211_11',
 'j21_3',
 'j21_4',
 'j23',
 'j24',
 'j25',
 'j26',
 'j27',
 'j28',
 'j29',
 'j308',


In [14]:
for col in tqdm(cols_replacement):
    tmp = variable_value_labels_df[variable_value_labels_df['question'] == col]
    repl_dict = {x:y for x, y in zip(tmp['answer'], tmp['mapping'])}
    panel[col] = panel[col].map(repl_dict)
    
panel

  0%|          | 0/3141 [00:00<?, ?it/s]

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)


In [15]:
# panel = panel.replace([
#     99999998.0, 99999997.0, 99999996.0, 99999995.0, 
#     99999994.0, 99999993.0, 99999992.0, 88888888.0
# ], np.nan)

In [16]:
panel['j60'].max()

6116000.0

In [17]:
panel.to_parquet('../data/RLMS_IND_2015_2024_eng_prepared.parquet', index=False)

In [14]:
variable_value_labels_df.to_excel('../data/variable_dictionary.xlsx', index=False)